# The 4 commands to run CESM

Running CESM involves four main steps:

1. **Create** a case.
2. **Set up** the case.
3. **Build** the CESM executable.
4. **Submit** the simulation.

Each command performs a different task. Understanding these steps will make it much easier to modify your simulations and diagnose problems later.

## Before creating a case

You must make three choices:

| Choice | What it means |
|---|---|
| **Case name** | The name of your experiment |
| **Component set** (`COMPSET`) | Which model components and experimental conditions CESM will use |
| **Grid** (`GRID`) | The horizontal resolution used by the model components |

The general form of the workflow is shown below.

:::{important}
`CASE_NAME`, `COMPSET`, and `GRID` are placeholders. Do not enter them literally. Replace them with the values selected for your experiment.
:::

## Step 1: Create the case

Begin in the `cime/scripts` directory of the CESM code:

```bash
cd ~/code/release-cesm2.1.5/cime/scripts
```

The general form of the command is:

```text
./create_newcase \
    --case ~/cases/CASE_NAME \
    --compset COMPSET \
    --res GRID
```

This command creates a new **case directory** containing the configuration for your experiment.

At this stage, CESM knows:

- The name of the experiment
- Which model components to use
- Which grid to use
- Where the case, build/run, and archive directories will be located

The model has not yet been built or run.

## Step 2: Set up the case

Move into the case directory:

```text
cd ~/cases/CASE_NAME
```

Then run:

```bash
./case.setup
```

`case.setup` prepares the scripts and configuration files that CESM needs to build and run the experiment.

The model has still not been compiled or run.

## Step 3: Build CESM

Build the model executable with:

```bash
qcmd -- ./case.build
```

Building means compiling the CESM source code into an executable program that Derecho can run.

The `qcmd` command asks Derecho to perform the build on a compute node. Building CESM is computationally intensive and should not be done directly on a login node.

A successful build produces the CESM executable in the **build directory**.

:::{note}
You normally build a case only once. You may need to rebuild it later if you change the source code or certain configuration settings.
:::

## Step 4: Submit the simulation

Submit the simulation with:

```bash
./case.submit
```

This command sends the simulation to Derecho's PBS batch scheduler.

The model does not necessarily begin running immediately. It may wait in the queue until the requested computing resources become available.

When the job starts, CESM runs in the **run directory**.

## Follow the job

Use `qstat` to see whether your job is queued or running:

```bash
qstat -u $USER
```

You may see output similar to this:

```text
                                                            Req'd  Req'd   Elap
Job ID          Username Queue    Jobname    SessID NDS TSK Memory Time  S Time
--------------- -------- -------- ---------- ------ --- --- ------ ----- - -----
6315297.desche* hannay   cpu      run.case01    --    6 768  1410g 12:00 Q   --
6315298.desche* hannay   cpu      st_archiv*    --    1   1  235gb 00:20 H   --
```

CESM submits two jobs: `run.CASE_NAME` runs the simulation, and `st_archive` organizes the output after the simulation finishes.

The `S` column shows the job status. The most common status codes are:

| Status | Meaning |
|---|---|
| `Q` | The job is queued and waiting to run. |
| `R` | The job is running. |
| `H` | The job is being held while it waits for another job to finish. |
| Not listed | The job has finished or stopped because of an error. |

A job disappearing from `qstat` does not necessarily mean that it succeeded.

## Check whether the simulation succeeded

### 1. Check `CaseStatus`

CESM records the status of each step in a file named `CaseStatus`, located in the case directory.

Use:

```text
tail -20 ~/cases/CASE_NAME/CaseStatus
```

A successful simulation includes:

```text
case.run success
```

After CESM organizes the output files, you should also see:

```text
st_archive success
```

If you see `case.run error`, the simulation crashed. The log files will provide more information about what went wrong.

### 2. Check the CESM log

You can also search the main CESM log for a successful-termination message:

```text
zgrep "SUCCESSFUL TERMINATION" ~/cases/CASE_NAME/logs/cesm.log.*.gz
```

A successful run should contain a line similar to:

```text
(seq_mct_drv): =============== SUCCESSFUL TERMINATION OF CPL7-cesm ===============
```

If you do not find this message, examine the log files for an error or traceback. The [Troubleshooting](../09.Troubleshooting/9.1_log_files_and_crashes.ipynb) chapter explains what to check when a run crashes.

### 3. Check the archive

After `st_archive success` appears, list the archive directory:

```text
ls /glade/derecho/scratch/$USER/archive/CASE_NAME
```

You should see directories containing model output, restart files, and log files.

## How the five directories are used

| Directory | Role in the workflow |
|---|---|
| **CESM code** | Provides the source code and the `create_newcase` command. |
| **Input data** | Provides the grids, initial conditions, and forcing data required by the selected case. |
| **Case** | Is created by `create_newcase` and contains the experiment configuration and commands. |
| **Build/run** | Is used when CESM is compiled and when the simulation runs. |
| **Archive** | Receives organized output, restart, and log files after the run. |

![The CESM directories after a run completes](../../images/cesm_directories_workflow.png)

In the next exercise, you will apply this workflow to create and run a specific CESM case.